# 03. Model Training & Validation

This notebook covers training baseline, seasonality-based machine learning, and advanced deep learning models:

1. **Baseline Models**:
   - Naive (predicts $y_{t}$ for $y_{t+1}$)
   - Moving Average (24-hour rolling mean)
   - Linear Regression (on lag features only)

2. **Seasonality / ML Models**:
   - Linear Regression with Fourier Features
   - XGBoost Regressor (on lags, calendar, and Fourier features)

3. **Advanced Model**:
   - PyTorch LSTM (using 24-hour lookback window sequences)

In [1]:
import sys
sys.path.append('../')

import os
import numpy as np
import pandas as pd
import xgboost as xgb
import torch

from src.models import NaiveModel, MovingAverageModel, train_linear_regression, PyTorchLSTMRegressor

## 1. Load Processed Splits

In [2]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (12076, 26)
Val shape: (2588, 26)
Test shape: (2588, 26)


## 2. Define Features & Target

In [3]:
target_col = 'OT'
raw_features = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL']
calendar_features = ['hour', 'dayofweek', 'month', 'is_weekend']
fourier_features = ['sin_24', 'cos_24', 'sin_168', 'cos_168']
lag_features = [col for col in train_df.columns if 'lag' in col]
rolling_features = [col for col in train_df.columns if 'roll' in col]

# Set targets
y_train = train_df[target_col]
y_val = val_df[target_col]
y_test = test_df[target_col]

## 3. Train Baseline Models

In [4]:
# Naive Model
naive_model = NaiveModel()
naive_preds = naive_model.predict(test_df)

# Moving Average Model (24-hour window)
ma_model = MovingAverageModel(window=24)
ma_preds = ma_model.predict(test_df)

# Linear Regression Baseline (on lag features only)
X_train_lr_base = train_df[lag_features]
X_test_lr_base = test_df[lag_features]
lr_base_model = train_linear_regression(X_train_lr_base, y_train)
lr_base_preds = lr_base_model.predict(X_test_lr_base)

## 4. Train Seasonality / ML Models

In [5]:
# Linear Regression with Fourier features
fourier_lr_features = lag_features + fourier_features
X_train_lr_fourier = train_df[fourier_lr_features]
X_test_lr_fourier = test_df[fourier_lr_features]
lr_fourier_model = train_linear_regression(X_train_lr_fourier, y_train)
lr_fourier_preds = lr_fourier_model.predict(X_test_lr_fourier)

# XGBoost Regressor (All engineered features)
all_features = raw_features + calendar_features + fourier_features + lag_features + rolling_features
X_train_xgb = train_df[all_features]
X_test_xgb = test_df[all_features]

xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train_xgb, y_train)
xgb_preds = xgb_model.predict(X_test_xgb)

## 5. Train Advanced Model (PyTorch LSTM)

In [6]:
# Helper function to convert dataframe data into sequence tensors
def create_sequences(df, target_col, feature_cols, seq_len=24):
    X, y = [], []
    feature_data = df[feature_cols].values
    target_data = df[target_col].values
    for i in range(len(df) - seq_len):
        X.append(feature_data[i : i + seq_len])
        y.append(target_data[i + seq_len])
    return np.array(X), np.array(y)

# Use raw + calendar + fourier + rolling features for LSTM inputs
lstm_features = raw_features + calendar_features + fourier_features + rolling_features
seq_len = 24

X_train_seq, y_train_seq = create_sequences(train_scaled_df := train_df, target_col, lstm_features, seq_len)
X_test_seq, y_test_seq = create_sequences(test_scaled_df := test_df, target_col, lstm_features, seq_len)

print("Train sequence shape:", X_train_seq.shape)
print("Train sequence target shape:", y_train_seq.shape)

# Fit PyTorch LSTM
device = "cuda" if torch.cuda.is_available() else "cpu"
lstm_regressor = PyTorchLSTMRegressor(
    input_dim=len(lstm_features),
    hidden_dim=32,
    num_layers=2,
    lr=0.005,
    epochs=5,
    batch_size=64,
    device=device
)
lstm_regressor.fit(X_train_seq, y_train_seq)

# Predict on test sequences
lstm_preds = lstm_regressor.predict(X_test_seq)

# Since sequences drop the first seq_len samples of the test set, we align other predictions and target
aligned_test_df = test_df.iloc[seq_len:].reset_index(drop=True)
y_test_aligned = y_test_seq

Train sequence shape: (12052, 24, 18)
Train sequence target shape: (12052,)


## 6. Save Predictions

In [7]:
predictions_df = pd.DataFrame({
    "date": aligned_test_df["date"],
    "y_true": y_test_aligned,
    "naive": naive_preds[seq_len:],
    "moving_average": ma_preds[seq_len:],
    "linear_regression_baseline": lr_base_preds[seq_len:],
    "linear_regression_fourier": lr_fourier_preds[seq_len:],
    "xgboost": xgb_preds[seq_len:],
    "lstm": lstm_preds
})

os.makedirs('../results', exist_ok=True)
predictions_df.to_csv('../results/predictions.csv', index=False)
print("Predictions saved successfully.")

Predictions saved successfully.
